<a href="https://colab.research.google.com/github/Yashika-Bayeen/agentic-ai/blob/main/Day_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced RAG with ChromaDB

In [ ]:
# Install dependencies
!pip install chromadb groq numpy python-dotenv -q
print("✅ Libraries installed.")

# 1. Setup and Baseline RAG

In [ ]:
import math
import re
from collections import Counter

CORPUS = [
    "A Machine Learning Engineer must be fluent in Python and PyTorch.",
    "MLOps skills include Docker, CI/CD, and model monitoring.",
    "Vector databases power retrieval for RAG systems.",
    "Frontend engineers focus on React and TypeScript."
]

def tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

def embed(text: str) -> Counter:
    return Counter(tokenize(text))

def cosine_similarity(a: Counter, b: Counter) -> float:
    common = set(a) & set(b)
    dot = sum(a[t] * b[t] for t in common)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

# Search
query = "Tell me about vector databases"
q_vec = embed(query)
for doc in CORPUS:
    score = cosine_similarity(q_vec, embed(doc))
    print(f"Similarity: {score:.3f} | '{doc}'")

In [ ]:
import chromadb

# Initialize database in-memory
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="roles_collection")

# Add documents
collection.add(
    documents=CORPUS,
    ids=[f"id_{i}" for i in range(len(CORPUS))]
)

# Query
results = collection.query(
    query_texts=["vector database and RAG"],
    n_results=1
)
print("Query Results:")
print(results['documents'][0])

In [ ]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def rag_pipeline(user_query):
    # 1. Retrieve
    results = collection.query(query_texts=[user_query], n_results=1)
    retrieved_chunk = results['documents'][0][0]

    # 2. Stuff Prompt
    system_prompt = (
        "You are an assistant. Answer the user's question USING ONLY the provided context. "
        "If the context doesn't contain the answer, say 'I don't know.'\n\n"
        f"[CONTEXT]\n{retrieved_chunk}"
    )

    # 3. Generate
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.0
    )
    return completion.choices[0].message.content

print("RAG Answer:")
print(rag_pipeline("What skills does an MLOps engineer need?"))

# 2. Advanced RAG Techniques

In [ ]:
def custom_text_splitter(text: str, chunk_size: int, overlap: int) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
        if end >= len(text):
            break
    return chunks

doc_text = "CareerForge is an advanced platform containing career mapping, curriculum alignment, and multi-agent systems."
split_chunks = custom_text_splitter(doc_text, chunk_size=30, overlap=10)

print(f"Document split into {len(split_chunks)} chunks:")
for idx, chunk in enumerate(split_chunks):
    print(f"  Chunk {idx}: '{chunk}' (Length: {len(chunk)})")

The `query_with_threshold` function demonstrates a key advanced RAG technique: **similarity thresholding**.

*   It queries the `collection` with a given `query_str`.
*   It retrieves the `distance` (a measure of dissimilarity) of the most relevant document.
*   If this `distance` exceeds a predefined `threshold`, it signifies that the retrieved document is not sufficiently similar to the query.
*   In such cases, the function proactively refuses to use the irrelevant context, instead returning a polite message indicating no relevant information was found. This prevents the LLM from hallucinating or generating answers based on loosely related or incorrect context.

In [ ]:
meta_collection = chroma_client.create_collection(name="meta_collection")

# Add chunks with metadata tags
meta_collection.add(
    documents=CORPUS,
    metadatas=[
        {"domain": "engineering"},
        {"domain": "operations"},
        {"domain": "infrastructure"},
        {"domain": "design"}
    ],
    ids=[f"meta_id_{i}" for i in range(len(CORPUS))]
)

# Query with domain = 'engineering'
filtered_results = meta_collection.query(
    query_texts=["skills and frameworks"],
    where={"domain": "engineering"},
    n_results=1
)

print("Filtered Query Result:")
print("Documents:", filtered_results['documents'][0])
print("Metadata:", filtered_results['metadatas'][0])